# NLP Final: lost-item candidate ranking

This notebook demonstrates a reproducible **fictional** Strathmore-themed exercise. A lost report is compared with a pool of found reports and the system returns candidates for staff review. A score does not verify ownership or authorize release. The 2026 office inventories supply aggregate category counts only; none of the individual records below came from those inventories.

## 1. Sources and dataset

The user transcribed two Office category tables. Their quantities total 388 for June–July and 299 for August. The linked original Google Docs were inaccessible to the connected account, so the transcription still needs checking against them. The controlled lost and found descriptions below were AI authored as fictional development examples; independent team review is pending.

In [1]:
from pathlib import Path
import csv, json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts/prepare_ranking_benchmark_v2.py').is_file()), Path.cwd())
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
assert (ROOT / 'scripts/prepare_ranking_benchmark_v2.py').exists(), 'Run from repository root'
from scripts.prepare_ranking_benchmark_v2 import main as prepare
from scripts.validate_ranking_dataset import validate
prepare()
DATA = ROOT / 'data/processed/ranking_v2'
audit = validate(DATA)
assert audit['valid'], audit['errors']
print(json.dumps(audit, indent=2))

{
  "version": "v2",
  "source_type": "AI-authored controlled fiction, inventory-informed categories",
  "previously_observed_v1_cases_used_for": "training and validation only",
  "new_held_out_test_file_sha256": "4d7f0b831ae2141316559d113bc41cd473d403158dd8fa87dcf4fe64196e6cb5",
  "item_groups_by_split": {
    "train": 16,
    "validation": 8,
    "test": 12
  },
  "found_count": 36,
  "lost_count": 43,
  "positive_labels": 36,
  "negative_labels": 428,
  "no_match_queries": 7,
  "candidate_pool_size": 36,
  "independent_review": "pending"
}
{
  "counts": {
    "found_reports.csv": 36,
    "lost_reports.csv": 43,
    "match_labels.csv": 464,
    "evaluation_queries.csv": 43
  },
  "label_counts": {
    "1": 36,
    "0": 428
  },
  "errors": [],
  "valid": true,
  "note": "Structural checks do not verify physical identity, independent authorship, or label truth."
}


## 2. Preparation and leakage control

Each controlled item has one found record and a separately worded lost claim. Some claims omit attributes, describe a colour differently, or name a different location. Same-category items create hard distractors. Version 1's observed cases are used only for development; 12 new item identities form the version 2 test split. No-match queries have no relevant found ID. The fixed candidate pool contains all 36 fictional found records.

In [2]:
def read_csv(name):
    with (DATA / name).open(encoding='utf-8', newline='') as stream:
        return list(csv.DictReader(stream))
found = read_csv('found_reports.csv'); lost = read_csv('lost_reports.csv')
labels = read_csv('match_labels.csv'); queries = read_csv('evaluation_queries.csv')
for split in ('train', 'validation', 'test'):
    groups = {r['item_group_id'] for r in found if r['split'] == split}
    print(split, 'item groups:', len(groups), 'matched queries:', sum(q['split']==split and q['has_match']=='1' for q in queries))
print('sources:', sorted({r['source_type'] for r in found}))
print('example:', {k: found[0][k] for k in ('found_id','category','description','source_type')})

train item groups: 16 matched queries: 16
validation item groups: 8 matched queries: 8
test item groups: 12 matched queries: 12
sources: ['ai_constructed_fictional']
example: {'found_id': 'f-g001', 'category': 'plastic_water_bottle', 'description': 'Clear plastic bottle with orange flip lid', 'source_type': 'ai_constructed_fictional'}


## 3. Original classification baseline

The retained CAT 2 Logistic Regression model predicts a binary pair label from engineered text and attribute comparisons. Here the same feature builder is fitted only on training pairs, then its output is used to rank each query's candidate pool. This makes it comparable with retrieval methods while preserving the earlier notebook and scripts.

In [3]:
from matching_system.feature_builder import FEATURE_NAMES, pair_features
from scripts.evaluate_ranking_benchmark import as_report
train_labels = [r for r in labels if r['split']=='train']
found_by_id = {r['found_id']: r for r in found}; lost_by_id = {r['lost_id']: r for r in lost}
example = train_labels[0]
features = pair_features(as_report(lost_by_id[example['lost_id']], 'lost'), as_report(found_by_id[example['found_id']], 'found'))
print('training pairs:', len(train_labels), 'features:', list(features))

training pairs: 256 features: ['description_similarity', 'category_match', 'category_conflict', 'colour_match', 'colour_conflict', 'brand_match', 'brand_conflict', 'model_match', 'model_conflict', 'material_match', 'material_conflict', 'distinctive_feature_similarity', 'location_similarity', 'time_difference_hours']


## 4. NLP text features and cosine similarity

TF-IDF gives more weight to terms that distinguish documents in the fitted corpus. Unigrams and bigrams retain individual words and short phrases. The vocabulary below is fitted only on training descriptions and then reused to transform validation, test and candidate records. Cosine similarity measures the angle between their feature vectors; it is textual relevance, not a probability of ownership.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from matching_system.nlp_retrieval import normalize_report_text
training_text = [normalize_report_text(r['description']) for r in found + lost if r['split']=='train']
vectorizer = TfidfVectorizer(ngram_range=(1,2), strip_accents='unicode').fit(training_text)
query_text = normalize_report_text(lost_by_id['l-g025']['description'])
found_text = normalize_report_text(found_by_id['f-g025']['description'])
similarity = cosine_similarity(vectorizer.transform([query_text]), vectorizer.transform([found_text]))[0,0]
print('vocabulary size:', len(vectorizer.vocabulary_), 'example cosine:', round(float(similarity), 4))

vocabulary size: 385 example cosine: 0.5094


## 5. Hybrid ranking and validation

The comparison script applies one candidate pool to keyword overlap, attribute scoring, Logistic Regression, TF-IDF cosine and a weighted hybrid. It selects a hybrid weight from 0.25, 0.50 or 0.75 using validation Recall@5 and MRR. A strong-suggestion cutoff is also derived from validation no-match scores. The test split is used only for reporting.

In [5]:
from scripts.evaluate_ranking_benchmark import main as evaluate
result_path = ROOT/'results/ranking_benchmark_v2.json'
evaluate(DATA, result_path)
results = json.loads(result_path.read_text(encoding='utf-8'))
print('selected TF-IDF weight:', results['hybrid']['tfidf_weight_selected_on_validation'])
print('training feature count:', len(results['logistic_regression']['features']))

{
  "keyword": {
    "matched_queries": 12,
    "no_match_queries": 4,
    "candidate_pool_size": 36,
    "top1_accuracy": 0.5,
    "recall_at_5": 0.9166666666666666,
    "mrr": 0.6422979797979799,
    "matched_details": [
      {
        "query_id": "q-g025",
        "first_rank": 1,
        "recall_at_5": 1.0
      },
      {
        "query_id": "q-g026",
        "first_rank": 3,
        "recall_at_5": 1.0
      },
      {
        "query_id": "q-g027",
        "first_rank": 4,
        "recall_at_5": 1.0
      },
      {
        "query_id": "q-g028",
        "first_rank": 1,
        "recall_at_5": 1.0
      },
      {
        "query_id": "q-g029",
        "first_rank": 1,
        "recall_at_5": 1.0
      },
      {
        "query_id": "q-g030",
        "first_rank": 1,
        "recall_at_5": 1.0
      },
      {
        "query_id": "q-g031",
        "first_rank": 11,
        "recall_at_5": 0.0
      },
      {
        "query_id": "q-g032",
        "first_rank": 2,
        "recall_at_5

## 6. Held-out ranking results

Top-1 accuracy asks whether the relevant found record is first. Recall@5 asks whether it appears in the first five. MRR averages the reciprocal of its first relevant rank. False suggestions count no-match queries whose top score exceeds the validation cutoff. All denominators are printed below. These are controlled-fiction results, not campus recovery rates.

In [6]:
print('method | top1 | recall@5 | MRR | no-match false suggestions')
for name, item in results['methods'].items():
    test = item['test']
    print(f"{name} | {test['top1_accuracy']:.3f} ({test['matched_queries']}) | {test['recall_at_5']:.3f} | {test['mrr']:.3f} | {item['test_no_match_false_suggestions']}/{item['test_no_match_denominator']}")
print('Hybrid relevant items above threshold:', results['methods']['hybrid']['test_relevant_found_above_threshold'], '/ 12')

method | top1 | recall@5 | MRR | no-match false suggestions
keyword | 0.500 (12) | 0.917 | 0.642 | 1/4
attributes | 0.167 (12) | 1.000 | 0.443 | 3/4
logistic_regression | 0.250 (12) | 1.000 | 0.549 | 4/4
tfidf | 0.167 (12) | 0.750 | 0.375 | 2/4
hybrid | 0.583 (12) | 1.000 | 0.764 | 4/4
Hybrid relevant items above threshold: 10 / 12


## 7. Explainability and application integration

The runtime `MatchingPipeline` returns lexical similarity alongside structured fact statuses and staff-workflow guidance. The cell below calls the FastAPI preview route function directly with a validated synthetic request. This checks its model-to-explanation path but does not test HTTP routing, the authenticated Firebase service, or Android connectivity. Ordinary users must not receive private verification details or treat recommendations as release decisions.

In [7]:
from pydantic import ValidationError
from matching_system.api import PreviewRequest, preview_matches
request = PreviewRequest(item_name='Metal water bottle',description='Blue bottle with a scratched lid',category_id='metal_water_bottle',location_id='msb',event_date='2026-11-10',limit=3)
body = preview_matches(request)
print('status:', body['status'], 'first ID:', body['matches'][0]['found_report_id'])
print('explanation:', body['matches'][0]['explanation'])
try:
    PreviewRequest(item_name='', description='')
except ValidationError:
    print('blank input: rejected')

status: possible_matches first ID: demo-found-metal
explanation: Facts: category agrees; location differs; event date agrees. Staff should review the conflicting details. Text similarity is lexical evidence, not a probability. Ownership is unverified; release is not authorized.
blank input: rejected


## 8. Interpretation and limits

Version 2 removes the automatic metadata copying that inflated version 1's scores. The hybrid ranks seven of twelve matched test queries first and all twelve within five, but its validation-selected cutoff incorrectly flags all four no-match queries as strong suggestions. Ranking and abstention therefore remain separate problems. These AI-authored fictional results do not establish real Strathmore accuracy. The team must independently review the cases and labels, check the original inventory documents, verify the authenticated service and Android flow, and approve this notebook before submission.